**Lab type:** review  
**Course:** EDA101 — Exploratory Data Analysis  
**Lesson:** Profiling a New Dataset  
**Task:** The AI-generated profiling notebook below runs without errors, but contains three issues: one type conversion mistake, one missing check, and one missingness interpretation that is incomplete. For each issue: identify what is wrong, explain why it matters, and fix the code.

## Setup: Load the dataset

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 84320

statuses = np.random.choice(
    ['delivered', 'pending', 'shipped', 'cancelled', 'processing', 'returned'],
    size=n,
    p=[0.729, 0.129, 0.097, 0.035, 0.010, 0.0005 + 0.0005]
)

order_dates = pd.date_range('2023-01-01', periods=n, freq='1min').strftime('%Y-%m-%d').tolist()

# delivery_date is null for all pending/processing rows (~8.1% of total)
delivery_dates = [
    None if s in ('pending', 'processing') else
    (pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(np.random.randint(1, 10)))).strftime('%Y-%m-%d')
    for s in statuses
]

return_flags = np.where(
    np.random.random(n) < 0.002,
    None,
    np.random.choice(['True', 'False'], n, p=[0.05, 0.95])
)

df = pd.DataFrame({
    'order_id':      ['ORD-{:05d}'.format(i) for i in range(1, n + 1)],
    'customer_id':   ['CUST-{:05d}'.format(np.random.randint(1, 31483)) for _ in range(n)],
    'order_date':    order_dates,
    'product_id':    ['PROD-{:03d}'.format(np.random.randint(1, 848)) for _ in range(n)],
    'quantity':      np.random.choice([1,2,3,4,5], n, p=[0.4,0.3,0.15,0.1,0.05]).astype(int),
    'unit_price':    np.round(np.random.choice([18.99, 29.99, 39.99, 49.99, 69.99, 99.99], n), 2),
    'discount':      np.round(np.random.choice([0.0, 0.05, 0.10, 0.15, 0.20, None], n, p=[0.1,0.2,0.3,0.2,0.17,0.03]), 2),
    'shipping_cost': np.round(np.random.uniform(0, 25, n), 2),
    'region':        np.random.choice(['North', 'South', 'East', 'West', 'Central', 'Pacific', 'Mountain', 'Northeast'], n),
    'channel':       np.random.choice(['web', 'mobile', 'in-store', 'phone'], n, p=[0.45, 0.35, 0.15, 0.05]),
    'status':        statuses,
    'delivery_date': delivery_dates,
    'return_flag':   return_flags,
    'revenue':       np.round(np.random.uniform(0, 500, n), 2),
})

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

---

## AI-generated profiling notebook

The cells below were generated by an AI assistant given the column names and a prompt to "profile this dataset". The code runs without errors. Your job is to find the three issues.

### Step 1 — Shape and column types

In [ ]:
# AI-generated
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)

### Step 2 — Missing values

In [ ]:
# AI-generated
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
summary = pd.DataFrame({'count': missing, 'pct': missing_pct})
print(summary[summary['count'] > 0])

### Step 3 — Numeric summaries

In [ ]:
# AI-generated
print(df.describe())

---

## Issue 1: Type conversion deferred

**What to find:** Look at the output of the `describe()` call above. Which columns are missing from the numeric summary that you would expect to see — or that should appear as dates rather than being silently excluded?

**Hint:** Check `df.dtypes` again. How are `order_date` and `delivery_date` stored? How is `return_flag` stored?

In [ ]:
# Detect: which columns that should be datetime or bool are stored as object?
print(df[['order_date', 'delivery_date', 'return_flag']].dtypes)

**Explanation:** Write your answer here — why does running `describe()` before type conversion cause you to miss information?

In [ ]:
# Fix: convert types before computing any summary statistics
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])
df['return_flag'] = df['return_flag'].map({'True': True, 'False': False})

print(df[['order_date', 'delivery_date', 'return_flag']].dtypes)
print("\nNumeric describe after type conversion:")
print(df.describe())

---

## Issue 2: Cardinality check missing

**What to find:** The AI-generated notebook profiled shape, missing values, and numeric summaries — but it skipped a check on the string columns entirely. Which check is missing, and why does it matter?

**Hint:** For each `object` column, how many unique values does it have? Are any of those counts surprising?

In [ ]:
# Add the missing check: unique value counts for all object columns
for col in df.select_dtypes('object').columns:
    n_unique = df[col].nunique()
    print(f"{col:<20} {n_unique:>6} unique")

**Explanation:** Write your answer here — what would you miss about data quality if you skipped cardinality checks on string columns?

Then for any low-cardinality columns (fewer than 20 unique values), inspect the actual values:

In [ ]:
# Inspect the actual values for low-cardinality object columns
for col in df.select_dtypes('object').columns:
    if df[col].nunique() < 20:
        print(f"\n{col}:")
        print(df[col].value_counts())

---

## Issue 3: Missingness reported but not interrogated

**What to find:** The AI notebook reported that `delivery_date` has missing values. That's correct — but reporting the count is not enough. Is the missingness random, or is it structurally explained by another column?

**Hint:** Cross-tabulate `delivery_date` missingness against `status`.

In [ ]:
# Detect: is delivery_date missingness explained by order status?
print(df[df['delivery_date'].isna()]['status'].value_counts())

**Explanation:** Write your answer here — what does this cross-tab tell you that the plain missing-value count did not? How would you treat this missing data differently knowing it is structural?

In [ ]:
# Confirm: what percentage of pending/processing rows are missing delivery_date?
for s in ['pending', 'processing']:
    subset = df[df['status'] == s]
    pct = subset['delivery_date'].isna().mean() * 100
    print(f"status='{s}': {pct:.1f}% missing delivery_date ({subset['delivery_date'].isna().sum()} of {len(subset)} rows)")

---

## Summary check

Run the corrected profiling pass in full — with type conversion first, cardinality checks included, and missingness interrogated:

In [ ]:
def profile(df):
    print("=== Shape ===")
    print(df.shape)

    print("\n=== Types ===")
    print(df.dtypes)

    print("\n=== Missing values ===")
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    print(pd.DataFrame({'count': missing, 'pct': missing_pct})[missing > 0])

    print("\n=== Cardinality (object columns) ===")
    for col in df.select_dtypes('object').columns:
        print(f"  {col:<20} {df[col].nunique():>6} unique")

    print("\n=== Numeric summary ===")
    print(df.describe())

profile(df)